<a href="https://colab.research.google.com/github/speediedan/interpretune/blob/main/src/it_examples/notebooks/publish/circuit_tracer_examples/ct_cross_backend_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# Uncomment to run installation steps if you do not have a development
# editable install and want to run this notebook in a fresh environment.
# %pip install uv
# %uv pip install --upgrade pip setuptools wheel && \
# %uv pip install 'git+https://github.com/speediedan/interpretune.git@main[examples]'
# %uv pip install --group git-deps
#
# NOTE: This cell is intentionally commented out. We will uncomment these
# install commands once we no longer need to preserve editable installs
# for active developer venvs.


# RTE Concept-Direction Analysis with Interpretune

This notebook demonstrates a two-phase concept-direction workflow for the RTE task using
Interpretune's analysis-op framework with the NNsight circuit-tracer backend
and real HuggingFace RTE data.

Key concepts demonstrated:
- Loading RTE data via RTEBoolqDataModule and test_dataloader
- Building per-pair concept-direction rows and storing them in AnalysisStore
- Aggregating a reusable entailment direction from stored concept rows
- Quality gate: projecting vocabulary onto the aggregated concept direction
- Validating direction alignment on real RTE batches
- Attribution graph generation and top-feature extraction on real examples
- Feature intervention on multiple real examples using the precomputed concept artifact

In [1]:
# Parameters - These will be injected by papermill during parameterized test runs
core_log_dir = None  # Analysis log directory; temp dir when unset
intervention_scale_factor = 30.0  # Multiplier for feature intervention amplitudes
n_top = 10  # Number of top features to extract
n_validation_batches = 3  # Number of dataloader batches to inspect

In [2]:
# @title Imports { display-mode: "form" }

import gc
import tempfile

import torch
from transformers import BatchEncoding

import interpretune as it
from it_examples import _ACTIVE_PATCHES  # noqa: F401
from it_examples.example_module_registry import MODULE_EXAMPLE_REGISTRY
from it_examples.utils.example_helpers import required_os_env
from it_examples.utils.nb_ui_utils import display_top_features_comparison, display_topk_token_predictions
from interpretune import AnalysisRunner, AnalysisRunnerCfg, ITSession, ITSessionConfig
from interpretune.analysis import execute_analysis_op
from interpretune.analysis.ops.base import AnalysisBatch
from interpretune.analysis.ops.helpers import resolve_embedding_weight, resolve_tokenizer
from interpretune.config import AnalysisCfg

In [3]:
# @title Environment Setup { display-mode: "form" }
env_path: str | None = None  # set to "/full/path/to/.env" to override
os_env_reqs = None
assert required_os_env(env_path=env_path, env_reqs=os_env_reqs)

In [4]:
# Load the Gemma-2 NNsight circuit-tracer demo configuration from the registry
base_itdm_cfg, base_it_cfg, dm_cls, m_cls = MODULE_EXAMPLE_REGISTRY.get("gemma2.rte_demo.circuit_tracer_nnsight")

if core_log_dir:
    base_it_cfg.core_log_dir = core_log_dir

# Use a lighter notebook runtime than the parity-oriented registry defaults.
base_it_cfg.entailment_mapping = ("▁Yes", "▁No")
base_it_cfg.nnsight_cfg.torch_dtype = torch.bfloat16
base_it_cfg.circuit_tracer_cfg.batch_size = 32
base_it_cfg.circuit_tracer_cfg.max_feature_nodes = 1024
base_it_cfg.circuit_tracer_cfg.intervention_value_source = "top_feature_activation_values"
base_it_cfg.circuit_tracer_cfg.intervention_scale_factor = intervention_scale_factor
base_it_cfg.circuit_tracer_cfg.transcoder_set = "gemma"

session_cfg = ITSessionConfig(
    adapter_ctx=(it.Adapter.core, it.Adapter.nnsight, it.Adapter.circuit_tracer),
    datamodule_cfg=base_itdm_cfg,
    module_cfg=base_it_cfg,
    datamodule_cls=dm_cls,
    module_cls=m_cls,
)

it_session = ITSession(session_cfg)
it.it_init(**it_session)
module = it_session.module
datamodule = it_session.datamodule
tokenizer = resolve_tokenizer(module)
yes_id, no_id = [int(token_id) for token_id in module.it_cfg.entailment_mapping_indices]


test_dl = datamodule.test_dataloader()

print(f"Model loaded: {type(module.replacement_model).__name__}")
print("Backend: NNsight circuit-tracer")
print(f"Dataset: aps/super_glue rte validation split (batch_size={test_dl.batch_size})")
print(f"Attribution runtime dtype: {base_it_cfg.nnsight_cfg.torch_dtype}")
print(f"Attribution batch_size: {base_it_cfg.circuit_tracer_cfg.batch_size}")
print(f"Attribution max_feature_nodes: {base_it_cfg.circuit_tracer_cfg.max_feature_nodes}")
print(f"Total batches available: {len(test_dl)}")

/home/speediedan/repos/interpretune/src/interpretune/config/transformer_lens.py:261: Interpretune manages the HF model instantiation via `model_name_or_path`. Since `tokenizer_name was not provided, the value provided for `tl_cfg.cfg.tokenizer_name` will be used for `tokenizer_name`.
/home/speediedan/repos/interpretune/src/interpretune/config/transformer_lens.py:261: Interpretune manages the HF model instantiation via `model_name_or_path`. Since `tokenizer_name` was provided, `tl_cfg.cfg.tokenizer_name` will be ignored.
/home/speediedan/repos/interpretune/src/interpretune/config/sae_lens.py:114: use_bridge=True is only meaningful when backend='transformerlens'. This setting will be ignored.
/home/speediedan/repos/interpretune/src/interpretune/config/shared.py:221: Could not find an auto-composition for <class 'interpretune.config.module.ITConfig'> that supports all of the following kwargs: {'circuit_tracer_cfg': CircuitTracerConfig(backend='transformerlens', model_name=None, transcoder

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[INFO] interpretune.utils.logging: NNsight ReplacementModel initialized for Circuit Tracer
INFO:interpretune.utils.logging:NNsight ReplacementModel initialized for Circuit Tracer
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'
INFO:interpretune.utils.logging:Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute 'quantization_config'
[INFO] interpretune.utils.logging: Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute '_pre_quantization_dtype'
INFO:interpretune.utils.logging:Attempted to clean a key that was not present, continuing without cleaning that key: 'Gemma2Config' object has no attribute '_pre_quantization_dtype'
[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule
INFO:interpretune.

Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/2490 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/277 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: Setting up datamodule: InterpretunableDataModule
INFO:interpretune.utils.logging:Setting up datamodule: InterpretunableDataModule
[INFO] interpretune.utils.logging: Setting up model: InterpretunableModule
INFO:interpretune.utils.logging:Setting up model: InterpretunableModule
[INFO] interpretune.utils.logging: initializing optimizers and schedulers: InterpretunableModule
INFO:interpretune.utils.logging:initializing optimizers and schedulers: InterpretunableModule
[INFO] interpretune.utils.logging: Input gradient requirements handled by circuit tracer internally.
INFO:interpretune.utils.logging:Input gradient requirements handled by circuit tracer internally.


Model loaded: NNSightReplacementModel
Backend: NNsight circuit-tracer
Dataset: aps/super_glue rte validation split (batch_size=2)
Attribution runtime dtype: torch.bfloat16
Attribution batch_size: 32
Attribution max_feature_nodes: 1024
Total batches available: 139


## Phase A: Build a reusable concept-direction artifact

This section shows how to turn cached activation rows from the validation dataloader into a reusable concept direction. The workflow focuses on:

- extracting answer-position residual activations from a shared cache key
- filtering those rows into concept-specific examples using labels and logit differences
- using `AnalysisStore` as the persistence boundary for reusable concept artifacts
- aggregating stored rows into a reusable concept direction through the shared analysis execution path
- validating the aggregated direction on `aps/super_glue` `rte` examples

In [6]:
# Build a reusable concept-direction artifact from cached answer-position activations
latent_examples_cfg = AnalysisCfg(
    name="latent_example_rows",
    target_op=[
        it.labels_to_ids,
        it.model_fwd_w_cache,
        it.logit_diffs_cache,
        it.extract_concept_latent_state,
        it.extract_concept_latent_examples,
    ],
    run_inputs={
        "concept_group_a_label_ids": [0],
        "concept_group_b_label_ids": [1],
        "concept_group_a_name": "entailment",
        "concept_group_b_name": "non_entailment",
        "concept_cache_key": "unembed.hook_in",
        "concept_correct_only": False,
        "concept_weight_by_logit_diff": True,
    },
    output_schema=it.extract_concept_latent_examples.output_schema,
    names_filter=lambda name: name == "unembed.hook_in",
    ignore_manual=True,
)

analysis_runner = AnalysisRunner(
    AnalysisRunnerCfg(
        it_session=it_session,
        analysis_cfgs=latent_examples_cfg,
        limit_analysis_batches=n_validation_batches,
        max_epochs=1,
        cache_dir=tempfile.mkdtemp(prefix="ct_nb_cache_"),
        op_output_dataset_path=tempfile.mkdtemp(prefix="ct_nb_store_"),
        ignore_manual=True,
    )
)
concept_store = analysis_runner.run_analysis()

concept_direction_cfg = AnalysisCfg(
    target_op=it.concept_direction,
    input_store=concept_store,
    ignore_manual=True,
    save_tokens=False,
)

cd_result = execute_analysis_op(
    module,
    analysis_batch=AnalysisBatch(
        concept_direction_mode="mean_difference",
        concept_label="Entailment aggregate: activation-space direction",
    ),
    analysis_cfg=concept_direction_cfg,
)

concept_direction = cd_result.concept_direction
concept_label = cd_result.concept_label
concept_direction_mode = cd_result.concept_direction_mode

print(f"Stored {len(concept_store.dataset)} concept rows from {n_validation_batches} batches")
print(f"Concept direction shape: {concept_direction.shape}")
print(f"Direction norm: {concept_direction.norm().item():.4f}")
print(f"Representative answer token IDs - Yes: {yes_id}, No: {no_id}")

group_names = []
for value in concept_store["concept_group_name"]:
    if isinstance(value, list):
        group_names.extend(str(item) for item in value)
    else:
        group_names.append(str(value))

print("Concept group counts:")
for group_name in sorted(set(group_names)):
    count = sum(1 for name in group_names if name == group_name)
    print(f"  - {group_name}: {count}")

[INFO] interpretune.utils.logging: Preparing data: InterpretunableDataModule
INFO:interpretune.utils.logging:Preparing data: InterpretunableDataModule


Map:   0%|          | 0/2490 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.
INFO:interpretune.utils.logging:The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.


Map:   0%|          | 0/277 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.
INFO:interpretune.utils.logging:The following columns  don't have a corresponding argument in `NNSightReplacementModel.forward` and have been ignored: hypothesis, idx, label, premise, sequences. If hypothesis, idx, label, premise, sequences are not expected by `NNSightReplacementModel.forward`, you can safely ignore this message.


...

0it [00:00, ?it/s]

[INFO] interpretune.utils.logging: Running analysis epoch end hooks: InterpretunableModule
INFO:interpretune.utils.logging:Running analysis epoch end hooks: InterpretunableModule


Saving the dataset (0/1 shards):   0%|          | 0/3 [00:00<?, ? examples/s]

[INFO] interpretune.utils.logging: Running analysis end hooks: InterpretunableModule
INFO:interpretune.utils.logging:Running analysis end hooks: InterpretunableModule
[INFO] interpretune.utils.logging: Running stage end hooks on IT module: InterpretunableModule
INFO:interpretune.utils.logging:Running stage end hooks on IT module: InterpretunableModule
[INFO] interpretune.utils.logging: Running stage end hooks on IT datamodule: InterpretunableDataModule
INFO:interpretune.utils.logging:Running stage end hooks on IT datamodule: InterpretunableDataModule


Stored 3 concept rows from 3 batches
Concept direction shape: torch.Size([2304])
Direction norm: 1.0000
Representative answer token IDs - Yes: 6287, No: 1307
Concept group counts:
  - entailment: 3
  - non_entailment: 3


In [7]:
# Quality gate: project the full vocabulary onto the concept direction
embed_weight = resolve_embedding_weight(module).float()
scores = embed_weight @ concept_direction.to(embed_weight.device)
k_quality = 15

topk_yes = torch.topk(scores, k_quality)
topk_no = torch.topk(-scores, k_quality)

print(f"Top {k_quality} tokens aligned with the positive direction:")
print(f"{'Rank':<6} {'Token':<20} {'Score':<10}")
print("-" * 36)
for i in range(k_quality):
    token_str = tokenizer.decode([topk_yes.indices[i].item()])
    print(f"{i + 1:<6} {repr(token_str):<20} {topk_yes.values[i].item():<10.4f}")

print(f"\nTop {k_quality} tokens aligned with the negative direction:")
print(f"{'Rank':<6} {'Token':<20} {'Score':<10}")
print("-" * 36)
for i in range(k_quality):
    token_str = tokenizer.decode([topk_no.indices[i].item()])
    print(f"{i + 1:<6} {repr(token_str):<20} {topk_no.values[i].item():<10.4f}")

print(f"\nConcept token scores: 'Yes' = {scores[yes_id].item():+.4f}, 'No' = {scores[no_id].item():+.4f}")
print(f"Score separation: {(scores[yes_id] - scores[no_id]).item():.4f}")

Top 15 tokens aligned with the positive direction:
Rank   Token                Score     
------------------------------------
1      'Security'           0.2152    
2      ' security'          0.1978    
3      ' Security'          0.1968    
4      'security'           0.1865    
5      ' SECURITY'          0.1859    
6      'SECURITY'           0.1805    
7      ' estekak'           0.1542    
8      'Election'           0.1397    
9      ' keamanan'          0.1394    
10     'Seguridad'          0.1375    
11     ' segurança'         0.1375    
12     ' sécurité'          0.1354    
13     ' peace'             0.1328    
14     ' Polsek'            0.1303    
15     'Peace'              0.1298    

Top 15 tokens aligned with the negative direction:
Rank   Token                Score     
------------------------------------
1      ' Christopher'       0.2118    
2      ' Reeve'             0.2062    
3      'Christopher'        0.1705    
4      'rungsseite'         0.1451    
5   

In [8]:
# Validate the direction on aps/super_glue rte examples from the dataloader
entailment_mapping = tuple(module.it_cfg.entailment_mapping)
replacement_model = module.replacement_model
validated_examples = []

print(f"Validating concept direction on {n_validation_batches} batches from test_dataloader:\n")

for batch_idx, batch in enumerate(test_dl):
    if batch_idx >= n_validation_batches:
        break

    labels = batch["labels"]
    label_list = labels.tolist() if isinstance(labels, torch.Tensor) else list(labels)
    input_ids = batch["input"]

    for ex_idx in range(input_ids.shape[0]):
        prompt = tokenizer.decode(input_ids[ex_idx].tolist(), skip_special_tokens=True)
        logits, _ = replacement_model.get_activations(prompt)
        last_logits = logits.squeeze(0)[-1].float().cpu()

        yes_logit = last_logits[yes_id].item()
        no_logit = last_logits[no_id].item()
        gap = yes_logit - no_logit
        predicted_label = 0 if gap > 0 else 1
        true_label = label_list[ex_idx]
        is_correct = predicted_label == true_label

        validated_examples.append(
            {
                "batch_idx": batch_idx,
                "prompt": prompt,
                "label": entailment_mapping[true_label],
                "predicted": entailment_mapping[predicted_label],
                "yes_logit": yes_logit,
                "no_logit": no_logit,
                "gap": gap,
                "is_correct": is_correct,
            }
        )

n_correct = sum(1 for ex in validated_examples if ex["is_correct"])
n_total = len(validated_examples)

print(f"{'Batch':<7} {'Label':<6} {'Pred':<6} {'Yes-No gap':>12} {'Correct':<8} {'Prompt (truncated)'}")
print("-" * 95)
for ex in validated_examples:
    mark = "✓" if ex["is_correct"] else "✗"
    print(
        f"{ex['batch_idx']:<7} {ex['label']:<6} {ex['predicted']:<6} "
        f"{ex['gap']:>+12.4f} {mark:<8} {ex['prompt'][:50]}..."
    )

print(f"\nAccuracy: {n_correct}/{n_total} ({100 * n_correct / n_total:.1f}%)")
correct_examples = [ex for ex in validated_examples if ex["is_correct"]]
print(f"Correct examples available for intervention: {len(correct_examples)}")

Validating concept direction on 3 batches from test_dataloader:

Batch   Label  Pred     Yes-No gap Correct  Prompt (truncated)
-----------------------------------------------------------------------------------------------
0       ▁No    ▁Yes        +0.7500 ✗        Dana Reeve, the widow of the actor Christopher Ree...
0       ▁Yes   ▁Yes        +1.0000 ✓        Yet, we now are discovering that antibiotics are l...
1       ▁No    ▁Yes        +0.5000 ✗        Cairo is now home to some 15 million people - a bu...
1       ▁No    ▁Yes        +0.8750 ✗        The Amish community in Pennsylvania, which numbers...
2       ▁Yes   ▁Yes        +0.5000 ✓        Security forces were on high alert after an electi...
2       ▁Yes   ▁Yes        +0.8750 ✓        In 1979, the leaders signed the Egypt-Israel peace...

Accuracy: 3/6 (50.0%)
Correct examples available for intervention: 3


---
## Phase B — Attribution And Intervention From The Stored Latent Artifact

Using the aggregated latent concept artifact from Phase A, we run the circuit-tracer
ops directly on correctly answered `aps/super_glue` `rte` examples.

This phase demonstrates:
1. Reusing the stored aggregate concept direction instead of recomputing latent rows per example
2. Attribution graph generation for dataset examples from the dataloader
3. Top feature extraction and comparison across multiple graphs
4. Feature intervention with a notebook-safe attribution budget

The notebook intentionally uses the shortest correct prompts from the sampled
batches so the Gemma-2 NNsight circuit-tracer workflow remains executable on a
single GPU.

In [9]:
# Run circuit-tracer ops using the precomputed aggregate latent concept artifact
selected = sorted(correct_examples, key=lambda item: len(item["prompt"]))[:2]
print(f"\nSelected {len(selected)} shortest correct examples for intervention:\n")

shared_concept_batch = AnalysisBatch(
    concept_direction=concept_direction,
    concept_label=concept_label,
    concept_direction_mode=concept_direction_mode,
    concept_group_a_token_ids=[yes_id],
    concept_group_b_token_ids=[no_id],
)

empty_batch = BatchEncoding({})
intervention_results = []
for i, ex in enumerate(selected):
    print(f"--- Example {i + 1}: Expected {ex['label']!r} ---")
    print(f"  Prompt length: {len(ex['prompt'])} characters")
    print(f"  Prompt: {ex['prompt'][:80]}...")

    graph_batch = it.compute_attribution_graph(
        module,
        AnalysisBatch(**shared_concept_batch, prompts=[ex["prompt"]]),
        empty_batch,
        0,
        batch_size=4,
        max_feature_nodes=256,
    )
    feature_batch = it.extract_top_features(
        module,
        graph_batch,
        empty_batch,
        0,
        top_n=min(n_top, 5),
    )
    result = it.feature_intervention_forward(
        module,
        feature_batch,
        empty_batch,
        0,
        intervention_scale_factor=intervention_scale_factor,
    )

    pre_logits = result.pre_intervention_logits.float().cpu()
    post_logits = result.post_intervention_logits.float().cpu()
    pre_gap = (pre_logits[yes_id] - pre_logits[no_id]).item()
    post_gap = (post_logits[yes_id] - post_logits[no_id]).item()

    print(f"  Yes−No gap: pre={pre_gap:+.4f}  post={post_gap:+.4f}  change={post_gap - pre_gap:+.4f}")

    intervention_results.append(
        {
            "example": ex,
            "result": result,
            "pre_logits": pre_logits,
            "post_logits": post_logits,
            "pre_gap": pre_gap,
            "post_gap": post_gap,
        }
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nAll {len(intervention_results)} interventions complete.")

Phase 0: Precomputing activations and vectors



Selected 2 shortest correct examples for intervention:

--- Example 1: Expected '▁Yes' ---
  Prompt length: 303 characters
  Prompt: Security forces were on high alert after an election campaign in which more than...


Precomputation completed in 1.31s
Found 64929 active features
Phase 1: Running forward pass
Forward pass completed in 0.66s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0037
Will include 1024 of 64929 feature nodes
Input vectors built in 0.68s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 0.15s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:02<00:00, 371.84it/s]
Feature attributions completed in 2.76s
Attribution completed in 8.67s
Phase 0: Precomputing activations and vectors


  Yes−No gap: pre=+0.5000  post=+0.5000  change=+0.0000
--- Example 2: Expected '▁Yes' ---
  Prompt length: 338 characters
  Prompt: Yet, we now are discovering that antibiotics are losing their effectiveness agai...


Precomputation completed in 0.57s
Found 67490 active features
Phase 1: Running forward pass
Forward pass completed in 0.20s
Phase 2: Building input vectors
Using 1 custom attribution targets with total weight 0.0197
Will include 1024 of 67490 feature nodes
Input vectors built in 0.68s
Phase 3: Computing logit attributions
1 logit attribution(s) completed in 0.07s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1024/1024 [00:02<00:00, 385.09it/s]
Feature attributions completed in 2.66s
Attribution completed in 7.28s


  Yes−No gap: pre=+1.0000  post=+1.1250  change=+0.1250

All 2 interventions complete.


In [10]:
# Display intervention results
key_tokens = [("Yes", yes_id), ("No", no_id)]

for i, ir in enumerate(intervention_results):
    ex = ir["example"]
    print(f"\n{'=' * 60}")
    print(f"Example {i + 1}: Expected {ex['label']!r} | Batch {ex['batch_idx']}")
    display_topk_token_predictions(
        ex["prompt"][:80] + "...",
        ir["pre_logits"],
        ir["post_logits"],
        tokenizer,
        k=5,
        key_tokens=key_tokens,
    )

feature_sets = {}
scores_sets = {}
for i, ir in enumerate(intervention_results):
    label = f"Ex {i + 1}: {ir['example']['label']}"
    result = ir["result"]
    feature_sets[label] = [tuple(f.tolist()) for f in result.top_feature_ids[:5]]
    scores_sets[label] = [s.item() for s in result.top_feature_scores[:5]]

display_top_features_comparison(feature_sets, scores_sets)

print("\nIntervention summary (concept direction amplification):")
print(f"{'Example':<12} {'Label':<6} {'Pre gap':>10} {'Post gap':>10} {'Change':>10}")
print("-" * 52)
for i, ir in enumerate(intervention_results):
    ex = ir["example"]
    delta = ir["post_gap"] - ir["pre_gap"]
    print(f"Ex {i + 1:<9} {ex['label']:<6} {ir['pre_gap']:>+10.4f} {ir['post_gap']:>+10.4f} {delta:>+10.4f}")


Example 1: Expected '▁Yes' | Batch 2


Token,Probability,Distribution
,0.375,37.5%
,0.292,29.2%
<eos>,0.045,4.5%
,0.035,3.5%
If,0.024,2.4%
Token,Probability,Distribution
,0.375,37.5%
,0.292,29.2%
<eos>,0.045,4.5%
,0.035,3.5%



Example 2: Expected '▁Yes' | Batch 0


Token,Probability,Distribution
,0.272,27.2%
,0.187,18.7%
<eos>,0.069,6.9%
,0.033,3.3%
If,0.033,3.3%
Token,Probability,Distribution
,0.274,27.4%
,0.188,18.8%
<eos>,0.069,6.9%
,0.033,3.3%


#,Node,Score
1,"(25, 47, 4717)",310.0000
2,"(25, 42, 4717)",292.0000
3,"(25, 19, 4717)",282.0000
4,"(25, 59, 4717)",282.0000
5,"(25, 18, 4717)",260.0000
#,Node,Score
1,"(25, 28, 4717)",288.0000
2,"(25, 60, 4717)",280.0000
3,"(25, 27, 4717)",272.0000
4,"(23, 27, 11023)",198.0000



Intervention summary (concept direction amplification):
Example      Label     Pre gap   Post gap     Change
----------------------------------------------------
Ex 1         ▁Yes      +0.5000    +0.5000    +0.0000
Ex 2         ▁Yes      +1.0000    +1.1250    +0.1250


In [11]:
# Cleanup
del module, it_session, session_cfg, datamodule, test_dl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e6:.1f} MB")
else:
    print("Cleanup complete (CPU mode)")

GPU memory after cleanup: 9696.7 MB


## Summary

This notebook demonstrated end-to-end concept-direction analysis on the RTE task
using real dataloader examples and a store-backed latent extraction workflow.

| Phase | What | Result |
|-------|------|--------|
| A | Latent example extraction | Correct-only SAE latent rows from real RTE batches |
| A | Artifact persistence | Stored latent rows in `AnalysisStore` with logit-diff weights |
| A | Concept aggregation | Aggregated stored rows into a reusable latent concept direction |
| A | Real-data validation | Checked direction alignment on sampled dataloader batches |
| B | Attribution pipeline | Generated attribution graphs on real RTE examples |
| B | Feature intervention | Amplified concept-aligned features and measured logit shifts |
| B | Cross-example comparison | Compared top features across multiple examples |

The full composite pipeline is:
model_fwd_w_cache_latent_models → logit_diffs_cache → extract_concept_latent_state →
extract_concept_latent_examples → concept_direction → compute_attribution_graph →
graph_node_influence → extract_top_features → feature_intervention_forward